# Notebook 0b — Build SNOMED IS-A Graph

Builds a `networkx.DiGraph` of SNOMED CT IS-A relationships from an RF2 release and saves it as a shared prerequisite.

**Required before:**
- `utils.py` → `ic_distances()` (Block A, notebook 3)
- `3-geometric-analysis.ipynb` → Block C (Poincaré training)

**Input**
- `sct2_Relationship_Snapshot_*.txt` from a SNOMED CT RF2 release (set `SNOMED_RF2_DIR` below)
- `data/embeddings-concept-openai/concepts.csv` — breast concept IDs used for scope restriction

**Output**
- `data/stage1/snomed_isa_graph.pkl` — `nx.DiGraph` restricted to the breast concept subgraph (nodes reachable upward from all breast concepts)

In [ ]:
# parameters
SNOMED_RF2_DIR = "../../data/common/SNOMED/target"   # symlink → active SNOMED RF2 release
DATA_DIR       = "../../data/stage1"
CONCEPTS_CSV   = "../../data/embeddings-concept-openai/concepts.csv"

In [ ]:
import glob
import os
import pickle

import networkx as nx
import pandas as pd

os.makedirs(DATA_DIR, exist_ok=True)

## 0b-1. Locate RF2 relationship file

In [ ]:
pattern = os.path.join(SNOMED_RF2_DIR, "**", "sct2_Relationship_Snapshot_*.txt")
matches = glob.glob(pattern, recursive=True)

if not matches:
    raise FileNotFoundError(
        f"No sct2_Relationship_Snapshot_*.txt found under {SNOMED_RF2_DIR}.\n"
        "Download a SNOMED CT RF2 release from https://www.snomed.org or your national release centre "
        "and set SNOMED_RF2_DIR to the extracted directory."
    )

rf2_path = matches[0]
print(f"Using: {rf2_path}")
if len(matches) > 1:
    print(f"Warning: {len(matches)} files matched — using the first. Set SNOMED_RF2_DIR more specifically if needed.")

## 0b-2. Load IS-A relationships

Filter to active (`active == 1`) IS-A relationships (`typeId == 116680003`). Each edge is `sourceId → destinationId` (child → parent).

In [ ]:
rel = pd.read_csv(rf2_path, sep="\t", dtype=str)
print(f"Total relationships: {len(rel):,}")

isa = rel[(rel["typeId"] == "116680003") & (rel["active"] == "1")]
print(f"Active IS-A relationships: {len(isa):,}")

## 0b-3. Build full IS-A graph

In [ ]:
G_full = nx.DiGraph()
G_full.add_edges_from(zip(isa["sourceId"], isa["destinationId"]))

print(f"Full graph — nodes: {G_full.number_of_nodes():,}, edges: {G_full.number_of_edges():,}")

## 0b-4. Restrict to breast concept subgraph

Keep only nodes that are in the breast concept set or are ontological ancestors of any breast concept. This keeps the pickle small — the full SNOMED IS-A graph has ~350k nodes; the breast hierarchy is a small fraction.

In [ ]:
concepts_df = pd.read_csv(CONCEPTS_CSV, dtype=str)
breast_ids = set(concepts_df["concept_id"].tolist())
print(f"Breast concepts: {len(breast_ids)}")

# Collect ancestors: in a child→parent DiGraph, nx.descendants follows edges forward
# which yields ontological ancestors (parents, grandparents, ...)
subgraph_nodes = set(breast_ids)
missing = []
for cid in breast_ids:
    if cid in G_full:
        subgraph_nodes |= nx.descendants(G_full, cid)
    else:
        missing.append(cid)

if missing:
    print(f"Warning: {len(missing)} breast concepts not found in RF2 graph (may be retired or from a different release).")

G = G_full.subgraph(subgraph_nodes).copy()
print(f"Restricted graph — nodes: {G.number_of_nodes():,}, edges: {G.number_of_edges():,}")

## 0b-5. Save

In [ ]:
out_path = os.path.join(DATA_DIR, "snomed_isa_graph.pkl")
with open(out_path, "wb") as f:
    pickle.dump(G, f)

print(f"Saved: {out_path}")